# I-JEPA Walkthrough: Understanding Joint Embedding Predictive Architectures

This notebook walks through the I-JEPA (Image JEPA) architecture — the foundation
for all JEPA variants. By understanding I-JEPA, you'll grasp the core principles
that extend to V-JEPA (video) and VL-JEPA (vision-language).

## Key Concepts
1. **Prediction in latent space** — not pixels, not tokens, but abstract representations
2. **Multi-block masking** — what you hide determines what the model learns
3. **EMA target encoder** — the teacher slowly follows the student
4. **No augmentations needed** — unlike contrastive learning (CLIP, SimCLR)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Check available device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

## 1. The Vision Transformer Encoder

I-JEPA uses a ViT (Vision Transformer) as its encoder.
Images are split into patches, embedded, and processed through transformer blocks.

In [ ]:
from src.ijepa.models.encoder import VisionTransformer

# Build a tiny ViT for demonstration
encoder = VisionTransformer(
    img_size=32,       # Small images for demo
    patch_size=8,      # 8x8 patches → 4x4 = 16 patches
    embed_dim=192,     # Embedding dimension
    depth=4,           # 4 transformer blocks
    num_heads=3,       # 3 attention heads
).to(device)

# Count parameters
num_params = sum(p.numel() for p in encoder.parameters())
print(f'Encoder parameters: {num_params:,}')
print(f'Number of patches: {encoder.num_patches}')
print(f'Embedding dimension: {encoder.embed_dim}')

In [ ]:
# Forward pass — full image
dummy_image = torch.randn(1, 3, 32, 32, device=device)
full_output = encoder(dummy_image)
print(f'Full encoder output: {full_output.shape}')  # (1, 16, 192)

# Forward pass — masked (only context patches)
context_indices = torch.tensor([[0, 1, 2, 3, 8, 9, 10, 11]], device=device)  # 8 of 16 patches
masked_output = encoder(dummy_image, mask_indices=context_indices)
print(f'Masked encoder output: {masked_output.shape}')  # (1, 8, 192)

## 2. Multi-Block Masking Strategy

The masking strategy is CRITICAL for I-JEPA. We mask out rectangular blocks
of patches (targets) and keep the rest (context). The predictor must predict
the target representations from the context.

Key insight: Target blocks must be large enough to require semantic understanding,
not just local texture completion.

In [ ]:
from src.ijepa.masks.multiblock import generate_masks

# Generate masks for a 4x4 grid of patches
context_idx, target_idx_list = generate_masks(
    batch_size=1,
    num_patches_h=4,
    num_patches_w=4,
    num_targets=2,          # 2 target blocks
    min_target_scale=0.15,  # Each target is 15-20% of patches
    max_target_scale=0.2,
)

print(f'Context patches: {context_idx[0].tolist()}')
print(f'Target block 1: {target_idx_list[0][0].tolist()}')
print(f'Target block 2: {target_idx_list[1][0].tolist()}')

# Visualize the mask
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
grid = np.zeros((4, 4))
for idx in context_idx[0].tolist():
    grid[idx // 4, idx % 4] = 1  # Context = blue
for i, tgt in enumerate(target_idx_list):
    for idx in tgt[0].tolist():
        grid[idx // 4, idx % 4] = 2 + i  # Targets = red/green

ax.imshow(grid, cmap='Set1')
ax.set_title('I-JEPA Masking\n(Blue=Context, Others=Targets)')
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.grid(True, linewidth=2)
plt.tight_layout()
plt.show()

## 3. The Predictor

The predictor is a smaller transformer that takes context embeddings
and predicts what the target embeddings should be.

Key design: The predictor is NARROWER than the encoder. This prevents
it from simply memorizing/copying and forces it to learn high-level
semantic relationships.

In [ ]:
from src.ijepa.models.predictor import JEPAPredictor

predictor = JEPAPredictor(
    num_patches=16,
    encoder_embed_dim=192,
    predictor_embed_dim=96,   # Half the encoder dim!
    depth=4,
    num_heads=3,
).to(device)

pred_params = sum(p.numel() for p in predictor.parameters())
print(f'Predictor parameters: {pred_params:,}')

# Predict target representations from context
context_emb = encoder(dummy_image, mask_indices=context_idx.to(device))
predicted_targets = predictor(
    context_emb,
    context_idx.to(device),
    target_idx_list[0].to(device),
)
print(f'Predicted target shape: {predicted_targets.shape}')

## 4. The Core JEPA Training Objective

The loss is simple: minimize the distance between predicted and actual
target representations. This is prediction in LATENT SPACE.

- **NOT pixel reconstruction** (like MAE)
- **NOT contrastive with negatives** (like CLIP)
- **Just: predict the abstract representation**

This forces the model to learn semantic features while ignoring
unpredictable surface-level details (exact textures, noise, etc.)

In [ ]:
import copy

# Create target encoder (EMA copy of context encoder)
target_encoder = copy.deepcopy(encoder)
target_encoder.requires_grad_(False)

# Get ground-truth target representations
with torch.no_grad():
    full_target_emb = target_encoder(dummy_image)
    actual_targets = torch.gather(
        full_target_emb,
        1,
        target_idx_list[0].to(device).unsqueeze(-1).expand(-1, -1, 192),
    )

# Compute loss: how close are predictions to reality?
loss = F.smooth_l1_loss(predicted_targets, actual_targets.detach())
print(f'JEPA loss: {loss.item():.4f}')
print(f'\nThis loss measures prediction quality in REPRESENTATION space.')
print(f'Lower = better at predicting what missing patches look like semantically.')

## 5. EMA Target Encoder Update

After each training step, the target encoder is updated as an
Exponential Moving Average (EMA) of the context encoder.

`target_params = momentum * target_params + (1 - momentum) * context_params`

The momentum starts at 0.996 and increases to 1.0 over training.
This provides a slowly-evolving target that stabilizes training.

In [ ]:
# EMA update demonstration
momentum = 0.996

# Before EMA: check distance between encoders
diff_before = sum(
    (p_t - p_c).abs().mean().item()
    for p_t, p_c in zip(target_encoder.parameters(), encoder.parameters())
)

# Simulate a gradient update on context encoder
fake_loss = encoder(dummy_image).mean()
fake_loss.backward()
with torch.no_grad():
    for p in encoder.parameters():
        if p.grad is not None:
            p.data -= 0.001 * p.grad
            p.grad.zero_()

# EMA update
with torch.no_grad():
    for p_t, p_c in zip(target_encoder.parameters(), encoder.parameters()):
        p_t.data.mul_(momentum).add_(p_c.data, alpha=1 - momentum)

diff_after = sum(
    (p_t - p_c).abs().mean().item()
    for p_t, p_c in zip(target_encoder.parameters(), encoder.parameters())
)

print(f'Param distance before EMA: {diff_before:.6f}')
print(f'Param distance after EMA:  {diff_after:.6f}')
print(f'\nTarget encoder slowly follows context encoder via EMA.')

## Summary: I-JEPA → V-JEPA → VL-JEPA

Everything you've seen here extends directly:

| Component | I-JEPA | V-JEPA | VL-JEPA |
|-----------|--------|--------|----------|
| Input | Image | Video | Image/Video + Text |
| Encoder | ViT | Video ViT | Frozen V-JEPA 2 |
| Predictor | Small Transformer | Small Transformer | Llama-3.2 layers |
| Target | EMA encoder | EMA encoder | EmbeddingGemma-300M |
| Loss | L2 in latent space | L2 in latent space | InfoNCE in latent space |
| Masking | Spatial blocks | Spatiotemporal tubes | N/A (text prediction) |

The core principle is always the same: **predict in abstract representation space**.

Next: `02_vjepa_video_features.ipynb` for video extension,
then `03_vljepa_inference_demo.ipynb` for the full vision-language model.